# Exercise 2.1: Supervised Learning with scikit-learn and PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/01_machine_learning/exercises/exercise_2_1_supervised_learning.ipynb)

This notebook is a small practical introduction to supervised learning. It uses synthetic datasets so that the expected model behavior is visible quickly.

You will practice:

- linear regression with `scikit-learn`
- gradient descent training with `torch`
- underfitting and overfitting with polynomial features
- logistic regression for binary classification
- basic regression and classification evaluation

The goal is not heavy mathematics. The goal is to see the workflow, change a few parameters, and understand what the model output means.


## 0. Setup

Run this first in Colab. The datasets are generated inside the notebook; no external data download is needed.


In [ ]:
%pip -q install scikit-learn pandas matplotlib seaborn torch


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import torch
from torch import nn

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


## 1. A tiny regression problem

We create a small housing-like example. The feature is apartment size. The target is monthly rent.

This is a supervised learning problem because every training row has an input `X` and a known answer `y`.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

n = 80
living_area = rng.uniform(20, 120, size=n)
rent = 320 + 11.5 * living_area + rng.normal(0, 95, size=n)

regression_df = pd.DataFrame({"living_area_m2": living_area, "monthly_rent_eur": rent})
display(regression_df.head())

fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=regression_df, x="living_area_m2", y="monthly_rent_eur", ax=ax)
ax.set_title("Small regression dataset")
plt.show()


## 2. Linear regression with scikit-learn

The usual workflow is:

1. split the data into train and test sets
2. fit the model on the train set
3. predict on the test set
4. evaluate with metrics


In [ ]:
X = regression_df[["living_area_m2"]]
y = regression_df["monthly_rent_eur"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
y_pred = linear_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Intercept: {linear_model.intercept_:.1f}")
print(f"Slope:     {linear_model.coef_[0]:.2f} EUR per m2")
print(f"MAE:       {mae:.1f} EUR")
print(f"RMSE:      {rmse:.1f} EUR")
print(f"R2:        {r2:.3f}")

x_line = np.linspace(X["living_area_m2"].min(), X["living_area_m2"].max(), 100).reshape(-1, 1)
y_line = linear_model.predict(x_line)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_train, y_train, label="train", alpha=0.75)
ax.scatter(X_test, y_test, label="test", alpha=0.9)
ax.plot(x_line, y_line, color="black", linewidth=2, label="linear regression")
ax.set_xlabel("Living area (m2)")
ax.set_ylabel("Monthly rent (EUR)")
ax.set_title("Linear regression fit")
ax.legend()
plt.show()


### Student task

Change `new_area` below and predict rent for a few apartment sizes. Does the prediction look reasonable for very small or very large values?


In [ ]:
new_area = 75
predicted_rent = linear_model.predict(pd.DataFrame({"living_area_m2": [new_area]}))[0]
print(f"Predicted rent for {new_area} m2: {predicted_rent:.0f} EUR")


## 3. Watching gradient descent with PyTorch

`LinearRegression()` solves the linear fit directly. Here we train the same kind of model by gradient descent.

The key idea: start with bad parameters, compute the error, then update the parameters step by step.


In [ ]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_x.fit_transform(X_train).astype("float32")
y_train_scaled = scaler_y.fit_transform(y_train.to_numpy().reshape(-1, 1)).astype("float32")
X_all_scaled = scaler_x.transform(X).astype("float32")

torch_model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(torch_model.parameters(), lr=0.08)
loss_fn = nn.MSELoss()

X_tensor = torch.tensor(X_train_scaled)
y_tensor = torch.tensor(y_train_scaled)

loss_history = []
snapshots = {}
epochs = 160
snapshot_epochs = {0, 1, 5, 20, 80, 159}

for epoch in range(epochs):
    optimizer.zero_grad()
    pred = torch_model(X_tensor)
    loss = loss_fn(pred, y_tensor)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    if epoch in snapshot_epochs:
        with torch.no_grad():
            y_all_scaled = torch_model(torch.tensor(X_all_scaled)).numpy()
            y_all = scaler_y.inverse_transform(y_all_scaled).ravel()
            snapshots[epoch] = y_all

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(loss_history)
axes[0].set_title("Training loss during gradient descent")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE loss on scaled train data")

order = np.argsort(X["living_area_m2"].to_numpy())
axes[1].scatter(X_train, y_train, alpha=0.45, label="train data")
for epoch, y_snapshot in snapshots.items():
    axes[1].plot(
        X["living_area_m2"].to_numpy()[order],
        y_snapshot[order],
        label=f"epoch {epoch}",
        alpha=0.85,
    )
axes[1].set_title("How the fitted line moves during training")
axes[1].set_xlabel("Living area (m2)")
axes[1].set_ylabel("Monthly rent (EUR)")
axes[1].legend()
plt.tight_layout()
plt.show()


### Student task

Try changing the learning rate `lr=0.08` to `0.005`, `0.3`, or `1.0`.

What changes in the loss curve? What happens if the learning rate is too small or too large?


## 4. Underfitting and overfitting

A model can be too simple or too flexible.

- Underfitting: the model misses the main pattern.
- Overfitting: the model follows noise and performs worse on new data.

We use polynomial features only to make the effect visible.


In [ ]:
rng = np.random.default_rng(7)
x = np.sort(rng.uniform(-3, 3, size=70))
y_curve = 0.8 * x**3 - 2.2 * x + rng.normal(0, 1.8, size=x.size)

X_curve = pd.DataFrame({"x": x})
y_curve_series = pd.Series(y_curve, name="target")

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_curve, y_curve_series, test_size=0.35, random_state=RANDOM_STATE
)

degrees = [1, 3, 15]
x_grid = np.linspace(-3.2, 3.2, 300).reshape(-1, 1)
results = []

fig, axes = plt.subplots(1, len(degrees), figsize=(15, 4), sharey=True)
for ax, degree in zip(axes, degrees):
    model = make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())
    model.fit(Xc_train, yc_train)

    train_pred = model.predict(Xc_train)
    test_pred = model.predict(Xc_test)
    train_rmse = mean_squared_error(yc_train, train_pred, squared=False)
    test_rmse = mean_squared_error(yc_test, test_pred, squared=False)
    results.append({"degree": degree, "train_RMSE": train_rmse, "test_RMSE": test_rmse})

    ax.scatter(Xc_train["x"], yc_train, label="train", alpha=0.7)
    ax.scatter(Xc_test["x"], yc_test, label="test", alpha=0.7)
    ax.plot(x_grid.ravel(), model.predict(x_grid), color="black", linewidth=2)
    ax.set_title(f"Polynomial degree {degree}")
    ax.set_xlabel("x")
    ax.set_ylabel("target")

axes[0].legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame(results))


### Student task

Change `degrees = [1, 3, 15]` to include values such as `2`, `5`, `10`, and `25`.

Which degree gives the best test RMSE? Which degree clearly overfits?


## 5. Logistic regression for classification

Now the target is a class, not a continuous number.

We create a tiny two-feature dataset. Imagine the target means whether a bike-sharing station is likely to have high demand during a time window.


In [ ]:
X_class, y_class = make_classification(
    n_samples=160,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.25,
    flip_y=0.06,
    random_state=RANDOM_STATE,
)

class_df = pd.DataFrame(
    X_class, columns=["distance_to_center_scaled", "near_poi_score_scaled"]
)
class_df["high_demand"] = y_class

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    class_df[["distance_to_center_scaled", "near_poi_score_scaled"]],
    class_df["high_demand"],
    test_size=0.30,
    stratify=class_df["high_demand"],
    random_state=RANDOM_STATE,
)

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    data=class_df,
    x="distance_to_center_scaled",
    y="near_poi_score_scaled",
    hue="high_demand",
    palette={0: "#1f77b4", 1: "#d62728"},
    ax=ax,
)
ax.set_title("Tiny binary classification dataset")
plt.show()


In [ ]:
logistic_model = make_pipeline(StandardScaler(), LogisticRegression())
logistic_model.fit(X_train_cls, y_train_cls)

y_pred_cls = logistic_model.predict(X_test_cls)
y_prob_cls = logistic_model.predict_proba(X_test_cls)[:, 1]

print(f"Accuracy: {accuracy_score(y_test_cls, y_pred_cls):.3f}")
print(classification_report(y_test_cls, y_pred_cls, target_names=["low demand", "high demand"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test_cls, y_pred_cls, display_labels=["low", "high"], ax=axes[0], colorbar=False
)
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test_cls, y_prob_cls, ax=axes[1])
axes[1].set_title("ROC curve")
plt.tight_layout()
plt.show()


In [ ]:
def plot_decision_boundary(model, X_data, y_data, title):
    x_min, x_max = X_data.iloc[:, 0].min() - 0.6, X_data.iloc[:, 0].max() + 0.6
    y_min, y_max = X_data.iloc[:, 1].min() - 0.6, X_data.iloc[:, 1].max() + 0.6
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 220), np.linspace(y_min, y_max, 220))
    grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=X_data.columns)
    proba = model.predict_proba(grid)[:, 1].reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(6, 5))
    contour = ax.contourf(xx, yy, proba, levels=20, cmap="RdBu_r", alpha=0.55)
    ax.contour(xx, yy, proba, levels=[0.5], colors="black", linewidths=2)
    sns.scatterplot(
        x=X_data.iloc[:, 0], y=X_data.iloc[:, 1], hue=y_data,
        palette={0: "#1f77b4", 1: "#d62728"}, edgecolor="white", ax=ax
    )
    ax.set_title(title)
    fig.colorbar(contour, ax=ax, label="Predicted probability of high demand")
    plt.show()

plot_decision_boundary(logistic_model, X_test_cls, y_test_cls, "Logistic regression decision boundary")


### Student task

The default decision threshold is `0.5`. Try changing the threshold below.

A lower threshold predicts more `high_demand` cases. A higher threshold predicts fewer. Watch how the confusion matrix changes.


In [ ]:
threshold = 0.50
y_pred_custom = (y_prob_cls >= threshold).astype(int)

print(f"Accuracy at threshold {threshold}: {accuracy_score(y_test_cls, y_pred_custom):.3f}")
print(confusion_matrix(y_test_cls, y_pred_custom))
print(classification_report(y_test_cls, y_pred_custom, target_names=["low demand", "high demand"]))


## 6. Logistic regression training with PyTorch

This is the same idea as the earlier gradient descent example, but now the loss is for binary classification.

We track both loss and accuracy so the training process is visible.


In [ ]:
cls_scaler = StandardScaler()
X_train_cls_scaled = cls_scaler.fit_transform(X_train_cls).astype("float32")
X_test_cls_scaled = cls_scaler.transform(X_test_cls).astype("float32")

X_train_t = torch.tensor(X_train_cls_scaled)
y_train_t = torch.tensor(y_train_cls.to_numpy().reshape(-1, 1).astype("float32"))
X_test_t = torch.tensor(X_test_cls_scaled)
y_test_np = y_test_cls.to_numpy().reshape(-1, 1)

torch_logistic = nn.Linear(2, 1)
optimizer = torch.optim.SGD(torch_logistic.parameters(), lr=0.12)
loss_fn = nn.BCEWithLogitsLoss()

history = []
epochs = 220

for epoch in range(epochs):
    optimizer.zero_grad()
    logits = torch_logistic(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0 or epoch == epochs - 1:
        with torch.no_grad():
            train_prob = torch.sigmoid(torch_logistic(X_train_t)).numpy()
            test_prob = torch.sigmoid(torch_logistic(X_test_t)).numpy()
        train_acc = ((train_prob >= 0.5).astype(int) == y_train_t.numpy()).mean()
        test_acc = ((test_prob >= 0.5).astype(int) == y_test_np).mean()
        history.append({"epoch": epoch, "loss": loss.item(), "train_accuracy": train_acc, "test_accuracy": test_acc})

history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["loss"])
axes[0].set_title("Logistic regression loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss")

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="train")
axes[1].plot(history_df["epoch"], history_df["test_accuracy"], label="test")
axes[1].set_title("Accuracy during training")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

display(history_df.tail())


### Student task

Try changing the PyTorch logistic regression learning rate `lr=0.12`.

- If training is slow, what happens to the loss curve?
- If training is unstable, what happens to the loss curve?
- Does higher train accuracy always mean better test accuracy?


## 7. Short evaluation checklist

Use these questions when you train a supervised model:

1. What is the target variable? Is it continuous or categorical?
2. What features are available at prediction time?
3. Did you keep a test set separate from training?
4. For regression: are MAE, RMSE, and R2 acceptable for the use case?
5. For classification: what errors are shown by the confusion matrix?
6. Does the model underfit, overfit, or generalize reasonably?

For geospatial AI, also ask whether the train and test split hides spatial leakage. Nearby places can be very similar, so random splits may look better than real deployment.
